# Making the data catalog
 - For more info, refer to Appendix

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
_p=Path(os.getcwd())
p=_p.parent

def parser(ln):
    prt = ln.split('|')
    if len(prt)<30:
        prt.extend([None]*(30- len(prt)))
    return prt

In [2]:
cat=pd.DataFrame()
test_id=[]      #
ref_id=[]       #
sample_id=[]    #
sex=[]          #
age=[]          #
level=[]        #
loading=[]      #
test=[]         #
ascii_id=[]     #
date=[]         #
cod=[]          #
injury=[]       #

for i in os.listdir(p/'raw/extracted'):
    dta=[]
    ld_flg=0
    asc_flg=0
    with open(p/str('raw/extracted/'+i+'/'+i+'.EV5'),'r') as fle:
        for lin in fle:
            lin=lin.strip()
            if lin[0:2]!='# ' and lin[0:2]!='--':
                dta.append(parser(lin))
    df=pd.DataFrame(dta)
    test_id.append(int(i[2:]))
    ref_id.append(df.iloc[0][6])
    sample_id.append(df.iloc[0][6][0:3])
    sex.append(df.iloc[1][1].lower())
    age.append(int(df.iloc[1][3]))
    level.append(str(df.iloc[0][6][4:6]))
    if 'DEG' in list(df[5]):
        deg=list(df[5]).index('DEG')
        asc_flg=1
        chk=pd.read_csv(p/str('raw/extracted/'+str(i)+'/'+str(i)+'.00'+str(df.iloc[deg][0])),sep='\t',header=None)
        if 'FLEXIBILITY' in df.iloc[0][2]:
            if(np.mean(chk[0])!=0):
                loading.append('both')
                ld_flg=1
        else:
            if(np.mean(chk[0])<0):
                loading.append('extension')
                ld_flg=1
            elif(np.mean(chk[0])>0):
                loading.append('flexion')
                ld_flg=1
    if 'FLEXIBILITY' in df.iloc[0][2]:
        test.append('flexibility')
        injury.append('NA')
    elif 'FAILURE' in df.iloc[0][2]:
        test.append('failure')
        if(int(i[2:])<7000):
            injury.append(df.iloc[2][7].lower())
        else:
            injury.append(df.iloc[3][7].lower())
    date.append(df.iloc[0][3].lower())
    if df.iloc[1][11]:
        cod.append(df.iloc[1][11].lower())
    else:
        cod.append('NA')
    if asc_flg==0 or ld_flg==0:
        ascii_id.append('NA')
    elif asc_flg==1:
        ascii_id.append(df.iloc[deg][0])
    if ld_flg==0:
        loading.append('NA')
lvl_map={
   '02':'C0C2',
   '03':'C0C3',
   '34':'C3C4',
   '45':'C4C5',
   '56':'C5C6',
   '67':'C6C7',
   '71':'C7T1',
   'O2':'C0C2'
}
level2=[lvl_map[i] if i in lvl_map else i for i in level]
cat['test_id']=test_id
cat['ref_id']=ref_id
cat['sample_id']=sample_id
cat['sex']=sex
cat['age']=age
cat['level']=level2
cat['loading']=loading
cat['test']=test
cat['ascii_id']=ascii_id
cat['date']=date
cat['cod']=cod
cat['injury']=injury
cat.to_csv(p/'clean-data/data-catalog.csv',index=False)

# Watermark

In [3]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pytensor,xarray

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install m2w64-toolchain`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


Last updated: Fri Apr 04 2025

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.25.0

pytensor: 2.23.0
xarray  : 2025.1.2

pathlib: 1.0.1
numpy  : 1.26.4
pandas : 2.2.2

Watermark: 2.5.0

